# Notebook 13 — Full Fine-Tuning (v3), all four experiments, unattended on 2 × T4

Runs E1–E4 with the **v3 full-backbone fine-tuning pipeline from Notebook 12, unchanged**: it pins the same commit (`bfdd972`, containing `src/train_vehicle_fullft.py` and `configs/full_finetuning.yaml`), the same frozen hash-verified splits, the same recipe, and the same run IDs (`E*-v3-run01`) and `v3_runs` folder. Only **how it runs** differs from Notebook 12:

- **Unattended:** start it with **Save Version → Save & Run All (Commit)**. It runs as a background job, so you can close the browser; the 1-hour inactivity shutdown applies only to interactive sessions.
- **Both GPUs:** GPU 0 runs E3 then E1 (Xception); GPU 1 runs E4 then E2 (ConvNeXt-Tiny). Each experiment still trains on one GPU at batch size 16, so the recipe is unchanged. The models run side by side on the same dataset, so any slowdown from sharing CPUs affects both equally.
- **Saved automatically:** everything in `/kaggle/working` is stored with the saved version: checkpoints, `metadata.json`, `history.csv`, test metrics, per-class test results, confusion matrices, one `.log` per run, a summary table, a v2-vs-v3 comparison table and validation-curve figures.

### Before you click Save & Run All
1. Settings: **Accelerator → GPU T4 x2**, **Internet → On**, and attach your private vehicle dataset with **Add Input** (the one used in Notebook 11, `sadiq17/vehicle-recognition-training-data`). Check it is listed under *Input* in the right-hand panel.
2. Leave the Run plan cell as it is (all four experiments, commit `bfdd972…` pre-filled).
3. **Save Version → Save & Run All (Commit) → Save**, then close the tab.

### While it runs / afterwards
- Progress is printed every 10 minutes: open the notebook → **Versions** (or *View active events*) → **Logs**.
- Expected time: roughly 2–3 hours (each GPU runs one Stanford run and one Pakistan run).
- When it finishes, open that version → **Output** → download `v3_runs/` (everything) or `v3_metrics_logs.zip` (small files only).
- If one run fails, the others still continue; the last cell then marks the version as failed and names the run, and its `.log` file shows why.

**Methodology note:** v3 results are exploratory and post-hoc (earlier test results were already inspected). Each run is tested once, after validation-based checkpoint selection; the code refuses to overwrite an existing `test_metrics.json`. The interrupted interactive E3 attempt in Notebook 12 saved no results and is not used.

## 0 — Run plan (the only cell you should need to edit)

In [ ]:
# ============================ RUN PLAN ============================
FROZEN_V3_COMMIT = 'bfdd972d4405df282d1ae65cfec1856a7b841a57'   # v3 code commit used by Notebook 12
V3_BRANCH = 'experiment/full-finetuning-v3'

# One queue per GPU; each queue runs its experiments one after another.
# Stanford runs first on both GPUs (E3 Xception | E4 ConvNeXt), then Pakistan (E1 | E2).
GPU_QUEUES = {0: ['E3', 'E1'], 1: ['E4', 'E2']}

RUN_ID_SUFFIX = 'v3-run01'      # run IDs become e.g. E4-v3-run01
RUN_POSTHOC_TEST = True         # single post-hoc test per run, right after its training
RUN_PREFLIGHT = True            # memory check + smoke test for each planned experiment
PROGRESS_EVERY_MIN = 10         # how often progress is printed to the Kaggle log
# ==================================================================
PLANNED = [e for q in GPU_QUEUES.values() for e in q]
assert PLANNED and len(PLANNED) == len(set(PLANNED)), f'Each experiment must appear once: {PLANNED}'
assert set(PLANNED) <= {'E1', 'E2', 'E3', 'E4'}, PLANNED
print('Planned experiments:', PLANNED)

## 1 — GPU check

In [ ]:
import os, sys, json, time, shutil, zipfile, threading, subprocess
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Settings → Accelerator → GPU T4 x2'
N_GPUS = torch.cuda.device_count()
print('Torch', torch.__version__, '| visible GPUs:', N_GPUS, '| CPU cores:', os.cpu_count())
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print('  GPU', i, p.name, round(p.total_memory / 1024**3, 2), 'GiB')
if N_GPUS < 2:
    print('WARNING: only one GPU visible. All planned experiments will run one after another on GPU 0.')
subprocess.run(['nvidia-smi'], check=True)

## 2 — Dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'pyyaml', 'scikit-learn', 'pandas', 'matplotlib'], check=True)
import timm, yaml, torchvision, sklearn, pandas as pd
print('timm', timm.__version__, '| torchvision', torchvision.__version__, '| sklearn', sklearn.__version__)

## 3 — Clone and pin the v3 commit
The repository and extracted images live in `/tmp`, **outside** `/kaggle/working`, so the ~2 GB dataset is not copied into the saved output. Only results go to `/kaggle/working/v3_runs`.

In [ ]:
REPO_URL = 'https://github.com/Batsy1857/vehicle-make-model-recognition.git'
assert len(FROZEN_V3_COMMIT) == 40 and all(c in '0123456789abcdef' for c in FROZEN_V3_COMMIT), \
    'Paste the actual 40-character v3 commit SHA into the Run plan cell first'
assert FROZEN_V3_COMMIT != '91e320a506749d01d141ff2115ecb585e03bce31', 'That is the frozen Notebook 11 commit'
REPO = Path('/tmp/vehicle-make-model-recognition')
OUTPUT = Path('/kaggle/working/v3_runs')
OUTPUT.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', V3_BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', FROZEN_V3_COMMIT], check=True)
sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert sha == FROZEN_V3_COMMIT
for rel in ['src/train_vehicle_fullft.py', 'configs/full_finetuning.yaml']:
    assert (REPO / rel).exists(), f'Missing v3 file: {rel}'
print('PINNED V3 CODE:', sha)

## 4 — Restore the exact prepared dataset
Kaggle normally **unpacks** an uploaded `data.zip` into folders, so this cell accepts either form:
- a folder containing both `pakistan/` and `stanford_cars/` (linked into the repository without copying, as in Notebook 11), or
- a still-zipped `data.zip` (extracted into the repository).

The split manifests and their SHA-256 hashes are verified in the next cell either way.

In [ ]:
INPUT = Path('/kaggle/input')

def find_data_sources(root, max_depth=6):
    folders, zips = [], []
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        p = Path(dirpath)
        depth = len(p.relative_to(root).parts)
        if 'pakistan' in dirnames and 'stanford_cars' in dirnames:
            folders.append(p)
            dirnames[:] = []            # no need to walk into the image folders
            continue
        zips += [p / f for f in filenames if f == 'data.zip']
        if depth >= max_depth:
            dirnames[:] = []
    return folders, zips

def show_input_tree(root, max_depth=4):
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        print('  ' * depth + (Path(dirpath).name or str(root)) + '/', sorted(dirnames)[:8], sorted(filenames)[:5])
        if depth >= max_depth:
            dirnames[:] = []

link = REPO / 'data'
if (link / 'pakistan').is_dir() and (link / 'stanford_cars').is_dir():
    print('Data already in place:', link.resolve())
else:
    folders, zips = find_data_sources(INPUT)
    if len(folders) == 1:
        assert not link.exists() or link.is_symlink(), f'Unexpected existing path: {link}'
        if link.is_symlink():
            link.unlink()
        link.symlink_to(folders[0], target_is_directory=True)
        print('Linked unpacked dataset folder:', folders[0])
    elif not folders and len(zips) == 1:
        print('Extracting', zips[0], '| size GiB', round(zips[0].stat().st_size / 1024**3, 2))
        with zipfile.ZipFile(zips[0]) as archive:
            names = archive.namelist()
            assert any(n.startswith('data/pakistan/') for n in names)
            assert any(n.startswith('data/stanford_cars/') for n in names)
            archive.extractall(REPO)
    else:
        print('Contents of /kaggle/input:')
        show_input_tree(INPUT)
        raise FileNotFoundError(
            f'Could not identify the dataset (folders found: {folders}; zips found: {zips}). '
            'Attach your private vehicle dataset with Add Input (it must contain pakistan/ and stanford_cars/), '
            'then save and run again.')
assert (link / 'pakistan').is_dir() and (link / 'stanford_cars').is_dir()
print('Data ready under', link, '->', link.resolve())

## 5 — Frozen manifest audit and recipe validation (no test images loaded)

In [ ]:
sys.path.insert(0, str(REPO / 'src'))
from vehicle_data import inspect_manifest
from train_vehicle_fullft import configure_v3_stage, epoch_lr   # import check only
cfg = yaml.safe_load((REPO / 'configs/experiments.yaml').read_text())
recipe = yaml.safe_load((REPO / 'configs/full_finetuning.yaml').read_text())
for name in ('pakistan', 'stanford_cars'):
    audit = inspect_manifest(REPO, name, check_files=True)
    assert audit['manifest_sha256'] == cfg['datasets'][name]['manifest_sha256']
    for split, expected in cfg['datasets'][name]['verified_split_counts'].items():
        observed = 'excluded_test_overlap' if name == 'pakistan' and split == 'excluded' else split
        assert audit['counts'].get(observed, 0) == expected
    print(name, audit['counts'], 'SHA PASS')
assert recipe['seed'] == 42
assert recipe['stages']['head_only']['epochs'] == 5
assert recipe['stages']['full_fine_tune']['epochs']['stanford_cars'] == 40
print('Frozen split + v3 recipe audit passed')

## 6 — Preflight: GPU-memory check and smoke test for each planned experiment
Same engineering checks as Notebook 12 (synthetic memory test; 2-batch smoke run). Not reportable results. A fresh batch session has an empty output folder, so the smoke run IDs do not clash.

In [ ]:
if RUN_PREFLIGHT:
    for exp in PLANNED:
        script = str(REPO / 'src/train_vehicle_fullft.py')
        print('MEMORY PREFLIGHT', exp, flush=True)
        subprocess.run([sys.executable, script, '--experiment', exp, '--run-id', f'{exp}-v3-memory',
                        '--output-dir', str(OUTPUT), '--mode', 'memory'], cwd=REPO, check=True)
        rid = f'{exp}-v3-smoke01'
        print('SMOKE', exp, flush=True)
        subprocess.run([sys.executable, script, '--experiment', exp, '--run-id', rid,
                        '--output-dir', str(OUTPUT), '--smoke'], cwd=REPO, check=True)
        m = json.loads((OUTPUT / rid / 'metadata.json').read_text())
        assert m['status'] == 'smoke_completed'
        assert m['parameters_full_fine_tune']['frozen'] == 0
        print('SMOKE OK:', rid, '| trainable', m['parameters_full_fine_tune']['trainable'], flush=True)
    print('Preflight passed for', PLANNED)
else:
    print('Preflight skipped')

## 7 — Train (and test) all planned experiments, one queue per GPU
Each experiment runs as its own process pinned to one GPU (`CUDA_VISIBLE_DEVICES`), with the same command as Notebook 12. Its full output goes to `v3_runs/<run-id>.log`; a short status line per run is printed here every few minutes.

In [ ]:
SCRIPT = str(REPO / 'src/train_vehicle_fullft.py')
queues = {g: q for g, q in GPU_QUEUES.items() if q} if N_GPUS >= 2 else {0: PLANNED}
for g in queues:
    assert g < max(N_GPUS, 1), f'GPU {g} is not available in this session'
status, lock = {}, threading.Lock()
for g, q in queues.items():
    for exp in q:
        status[exp] = {'gpu': g, 'state': 'waiting'}

def run_logged(cmd, gpu, log_path):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1')
    with open(log_path, 'a') as log:
        log.write('\n$ ' + ' '.join(cmd) + '\n'); log.flush()
        return subprocess.run(cmd, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT).returncode

def worker(gpu, exps):
    for exp in exps:                       # a failed run does not stop the next one in the queue
        rid = f'{exp}-{RUN_ID_SUFFIX}'
        log_path = OUTPUT / f'{rid}.log'
        base = [sys.executable, SCRIPT, '--experiment', exp, '--run-id', rid, '--output-dir', str(OUTPUT)]
        with lock:
            status[exp].update(state='training', log=log_path, start=time.time())
        code = run_logged(base, gpu, log_path)
        if code == 0 and RUN_POSTHOC_TEST:
            with lock:
                status[exp]['state'] = 'testing'
            code = run_logged(base + ['--mode', 'test', '--acknowledge-posthoc-test'], gpu, log_path)
        with lock:
            status[exp].update(state='done' if code == 0 else f'FAILED (exit {code})', end=time.time())

def last_line(path):
    if path is None:                       # experiment still waiting in its queue
        return ''
    try:
        lines = [l for l in Path(path).read_text(errors='ignore').splitlines() if l.strip()]
        return lines[-1][:150] if lines else '(no output yet)'
    except Exception:
        return '(log not readable yet)'

def report(t0):
    # Progress reporting must never stop training: any error here is printed and ignored.
    try:
        print(f'--- {time.strftime("%H:%M:%S")} | elapsed {(time.time() - t0) / 60:.0f} min ---', flush=True)
        with lock:
            snapshot = {exp: dict(s) for exp, s in status.items()}
        for exp, s in snapshot.items():
            print(f'  {exp} [GPU {s.get("gpu")}] {s.get("state", "?"):<10} {last_line(s.get("log"))}', flush=True)
    except Exception as exc:
        print('(progress report skipped:', type(exc).__name__, exc, ')', flush=True)

print('GPU queues:', queues, '| post-hoc test:', RUN_POSTHOC_TEST, flush=True)
t0 = time.time()
threads = [threading.Thread(target=worker, args=(g, q), daemon=True) for g, q in queues.items()]
for t in threads:
    t.start()
while any(t.is_alive() for t in threads):
    for _ in range(max(1, int(PROGRESS_EVERY_MIN * 60 / 5))):
        if not any(t.is_alive() for t in threads):
            break
        time.sleep(5)
    report(t0)
for t in threads:
    t.join()
FAILED = [e for e, s in status.items() if s['state'] != 'done']
print('\nALL QUEUES FINISHED in', round((time.time() - t0) / 60), 'min | failed:', FAILED or 'none', flush=True)

## 8 — Summary (validation selection + post-hoc test)

In [ ]:
EXPERIMENTS = {'E1': ('Pakistan', 'Xception'), 'E2': ('Pakistan', 'ConvNeXt-Tiny'),
               'E3': ('Stanford Cars', 'Xception'), 'E4': ('Stanford Cars', 'ConvNeXt-Tiny')}
rows = []
for exp in PLANNED:
    d = OUTPUT / f'{exp}-{RUN_ID_SUFFIX}'
    s = status.get(exp, {})
    row = {'experiment': exp, 'dataset': EXPERIMENTS[exp][0], 'model': EXPERIMENTS[exp][1],
           'status': s.get('state', 'not run'), 'gpu': s.get('gpu'),
           'wall_minutes': round((s['end'] - s['start']) / 60, 1) if 'end' in s and 'start' in s else None}
    if (d / 'metadata.json').is_file():
        meta = json.loads((d / 'metadata.json').read_text())
        row.update({k: meta.get(k) for k in ('best_validation_macro_f1', 'selected_stage', 'selected_epoch',
                                             'epochs_completed_finetune', 'total_train_seconds_excluding_validation',
                                             'peak_gpu_memory_mb', 'git_commit')})
        row['full_trainable_params'] = (meta.get('parameters_full_fine_tune') or {}).get('trainable')
    if (d / 'test_metrics.json').is_file():
        tm = json.loads((d / 'test_metrics.json').read_text()).get('test_metrics', {})
        row.update({f'test_{k}': tm.get(k) for k in ('accuracy', 'macro_f1', 'top5_accuracy', 'n')})
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT / 'v3_summary.csv', index=False)
pd.set_option('display.width', 250)
print(summary.to_string(index=False))

## 9 — Partial (v2, Notebook 11) vs full (v3) fine-tuning, and validation curves
The v2 numbers are the previously inspected Notebook 11 test results, copied here for reference. Figures go to `v3_runs/figures/`.

In [ ]:
V2_TEST = {  # Notebook 11 post-hoc test results (accuracy, macro-F1)
    'E1': (0.8082556591211718, 0.7472094887101349), 'E2': (0.8788282290279628, 0.8211063482252904),
    'E3': (0.5304066658375823, 0.5304583096355577), 'E4': (0.6664593955975625, 0.6624996896978089)}
cmp_rows = []
for _, r in summary.iterrows():
    v2_acc, v2_f1 = V2_TEST[r['experiment']]
    v3_acc, v3_f1 = r.get('test_accuracy'), r.get('test_macro_f1')
    has_v3 = v3_acc is not None and pd.notna(v3_acc)
    cmp_rows.append({'experiment': r['experiment'], 'dataset': r['dataset'], 'model': r['model'],
                     'v2_partial_test_acc_%': round(100 * v2_acc, 2),
                     'v3_full_test_acc_%': round(100 * v3_acc, 2) if has_v3 else None,
                     'change_acc_pts': round(100 * (v3_acc - v2_acc), 2) if has_v3 else None,
                     'v2_partial_macro_f1': round(v2_f1, 4),
                     'v3_full_macro_f1': round(v3_f1, 4) if has_v3 else None})
comparison = pd.DataFrame(cmp_rows)
comparison.to_csv(OUTPUT / 'v2_vs_v3_comparison.csv', index=False)
print(comparison.to_string(index=False))

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    FIG = OUTPUT / 'figures'
    FIG.mkdir(exist_ok=True)
    for dataset in ('Stanford Cars', 'Pakistan'):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for exp, (ds, model_name) in EXPERIMENTS.items():
            h_path = OUTPUT / f'{exp}-{RUN_ID_SUFFIX}' / 'history.csv'
            if ds != dataset or not h_path.is_file():
                continue
            h = pd.read_csv(h_path)
            x = range(1, len(h) + 1)
            axes[0].plot(x, h['val_macro_f1'], label=f'{model_name} ({exp})')
            axes[1].plot(x, h['val_accuracy'], label=f'{model_name} ({exp})')
        for ax, name in zip(axes, ('Validation macro-F1', 'Validation accuracy')):
            ax.axvline(5.5, color='gray', ls='--', lw=1, label='start of full fine-tuning')
            ax.set(title=f'{dataset}: {name}', xlabel='Epoch (head-only 1–5, then full fine-tuning)')
            ax.grid(alpha=0.25)
            ax.legend()
        fig.tight_layout()
        out_png = FIG / f'val_curves_{dataset.split()[0].lower()}.png'
        fig.savefig(out_png, dpi=150)
        plt.close(fig)
        print('Saved', out_png)
except Exception as exc:   # figures are a convenience; never fail the run because of them
    print('Figure generation skipped:', type(exc).__name__, exc)

## 10 — Package small files and finish

In [ ]:
archive_path = Path('/kaggle/working/v3_metrics_logs.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for f in OUTPUT.rglob('*'):
        if f.is_file() and f.suffix.lower() in ('.csv', '.json', '.log', '.png', '.npy'):
            z.write(f, arcname=f.relative_to(OUTPUT))
for d in sorted(OUTPUT.glob('E*-v3-*')):
    if d.is_dir():
        print(d.name, [(f.name, round(f.stat().st_size / 1024**2, 1)) for f in d.iterdir() if f.is_file()])
print('Small archive (metrics + logs):', archive_path)
print('All of /kaggle/working, including checkpoints, is saved with this version (Output tab).')
if FAILED:
    raise RuntimeError(f'These runs did not finish cleanly: {FAILED}. See v3_runs/<run-id>.log')